<a href="https://colab.research.google.com/github/git-Varun-hub/HCL-Ticket-project-/blob/main/Ticket_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# TICKET DATA ANALYSIS SYSTEM
import csv, os, random
import numpy as np
import pandas as pd

FILE_PATH = "tickets.csv"


departments = ("Technical Support", "Billing", "Account", "Delivery", "General")
categories = ["Login Issue", "Payment Problem", "Account Access", "Late Delivery", "Refund Request", "Product Query"]
priorities = {"Urgent": 4, "High": 3, "Medium": 2, "Low": 1}


def generate(path=FILE_PATH):
    names = ["Aarav", "Rohan", "Karan", "Rahul", "Ananya", "Diya", "Priya", "Neha", "Riya", "Kabir"]
    cities = ["Delhi", "Ghaziabad", "Noida", "Lucknow", "Jaipur"]
    channels = ["Email", "Phone", "Chat", "Web"]
    statuses = ["Open", "In Progress", "Resolved", "Closed"]
    random.seed(1)
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["ticket_id", "customer_name", "department", "category", "priority", "status", "channel", "city", "response_time", "resolution_time", "customer_rating"])
        for i in range(1, 61):
            row = [2000+i, random.choice(names), random.choice(departments), random.choice(categories), random.choice(list(priorities)), random.choice(statuses), random.choice(channels), random.choice(cities), random.randint(1,48), random.randint(1,120), random.randint(1,5)]
            if i in (10, 25, 40): row[8] = ""
            writer.writerow(row)


def add_derived_columns(df):
    df = df.copy()
    df["priority_score"] = df["priority"].map(priorities).fillna(0)
    def response_label(hours):
        if pd.isna(hours): return "Unknown"
        if hours <= 4: return "Fast"
        if hours <= 24: return "Medium"
        return "Slow"
    df["response_category"] = df["response_time"].apply(response_label)
    df["sla_status"] = np.where(df["response_time"] <= 24, "Within 24 Hours", "More Than 24 Hours")
    df.loc[df["response_time"].isna(), "sla_status"] = "Not Available"
    return df


def handle_missing_values(df):
    df = df.copy()
    missing = df.isnull().sum()
    print("Missing values before:")
    print(missing[missing > 0] if (missing > 0).any() else "No missing values")
    for col in ["response_time", "resolution_time", "customer_rating"]:
        if df[col].isnull().any(): df[col] = df[col].fillna(df[col].mean()).round(0)
    print("Missing numerical values filled with column mean.")
    return df


def numpy_demo(df):
    print("\n--- NumPy ---")
    arr1 = np.array(df["resolution_time"].dropna())
    arr2 = np.array(df[["response_time", "resolution_time", "customer_rating"]].dropna())
    print("1D array first 10:", arr1[:10])
    print("2D array shape:", arr2.shape)
    if arr2.size:
        print("Element at row 0, col 1:", arr2[0, 1])
        print("First 5 response times:", arr2[:5, 0])
        print("Rows 2 to 3:\n", arr2[2:4])
    for col in ["response_time", "resolution_time", "customer_rating"]:
        values = df[col].dropna().to_numpy()
        print(col, "-> sum:", values.sum(), "mean:", round(values.mean(),2), "max:", values.max(), "min:", values.min(), "std:", round(values.std(),2))


def pandas_demo(df):
    df = add_derived_columns(df)
    print("\n--- Selected columns (first 5 rows) ---")
    print(df[["ticket_id", "department", "priority", "status", "response_category"]].head())
    print("\n--- Rows 10 to 14 ---")
    print(df.iloc[10:15, 0:5])
    print("\n--- Filter: Urgent or High priority tickets ---")
    print(df[df["priority"].isin(["Urgent", "High"])][["ticket_id", "department", "category", "priority", "status"]].head(10))
    print("\n--- Top 5 tickets by resolution time ---")
    print(df.sort_values("resolution_time", ascending=False)[["ticket_id", "department", "resolution_time"]].head())
    print("\n--- Group by department ---")
    print(df.groupby("department")[["response_time", "resolution_time", "customer_rating"]].mean().round(2))
    print("\n--- Ticket count by status ---")
    print(df.groupby("status")["ticket_id"].count())
    print("\n--- Statistics ---")
    print(df[["response_time", "resolution_time", "customer_rating"]].describe().round(2))


def insights(df):
    df = add_derived_columns(df)
    dept_count = df.groupby("department")["ticket_id"].count()
    category_count = df["category"].value_counts()
    print("\n--- Insights ---")
    print("1. Busiest department:", dept_count.idxmax(), int(dept_count.max()), "tickets")
    print("2. Most common category:", category_count.idxmax(), int(category_count.max()), "tickets")
    print("3. High or urgent priority tickets:", int(df["priority"].isin(["Urgent", "High"]).sum()))
    print("4. Open tickets:", int((df["status"] == "Open").sum()))
    print("5. Resolved or closed tickets:", int(df["status"].isin(["Resolved", "Closed"]).sum()))
    print("6. Average response time:", round(df["response_time"].mean(),2), "hours")
    print("7. Average resolution time:", round(df["resolution_time"].mean(),2), "hours")
    print("8. Average customer rating:", round(df["customer_rating"].mean(),2), "out of 5")


def summary_report(df):
    df = add_derived_columns(df)
    print("\n--- Summary Report ---")
    print("Total tickets:", len(df))
    print("Unique customers:", df["customer_name"].nunique())
    print("Average response time:", round(df["response_time"].mean(),2), "hours")
    print("Average resolution time:", round(df["resolution_time"].mean(),2), "hours")
    print("Average customer rating:", round(df["customer_rating"].mean(),2), "out of 5")
    print("\nTickets per department:\n", df["department"].value_counts())
    print("\nTicket priorities:\n", df["priority"].value_counts())
    print("\nTicket statuses:\n", df["status"].value_counts())


def print_header(title): print("\n" + "="*60 + "\n" + title.center(60) + "\n" + "="*60)
def show_table(df): print(df.to_string(index=False) if len(df) else "No records to display")
def unique_values(records, key): return sorted({record[key] for record in records})
def list_methods_demo():
    ids = [101, 105, 110]
    ids.append(115); ids.extend([120,125]); ids.insert(0,100); ids.remove(110); ids.sort(reverse=True)
    return ids

def read_with_csv_module():
    with open(FILE_PATH, "r", newline="", encoding="utf-8") as file: return list(csv.DictReader(file))
def load_data(): return pd.read_csv(FILE_PATH)
def save_data(df): df.to_csv(FILE_PATH, index=False)
def next_ticket_id(df): return int(df["ticket_id"].max()) + 1 if len(df) else 2001
def add_record(df, record):
    df = pd.concat([df, pd.DataFrame([record])], ignore_index=True); save_data(df); return df
def search_record(df, keyword):
    keyword = str(keyword)
    return df[(df["ticket_id"].astype(str) == keyword) | df["customer_name"].str.contains(keyword, case=False, na=False) | df["category"].str.contains(keyword, case=False, na=False)]
def update_record(df, ticket_id, changes):
    if ticket_id not in df["ticket_id"].values: return df, False
    allowed = ["customer_name","department","category","priority","status","channel","city","response_time","resolution_time","customer_rating"]
    for column, value in changes.items():
        if column not in allowed: print("This field cannot be updated."); return df, False
        df.loc[df["ticket_id"] == ticket_id, column] = value
    save_data(df); return df, True
def delete_record(df, ticket_id):
    if ticket_id not in df["ticket_id"].values: return df, False
    df = df[df["ticket_id"] != ticket_id].copy(); save_data(df); return df, True


if not os.path.exists(FILE_PATH): generate(FILE_PATH)
records = read_with_csv_module()
print("Total records loaded:", len(records))
print("First record:", records[0])
print("Cities (set):", unique_values(records, "city"))
print("List methods demo:", list_methods_demo())
df = load_data()

while True:
    print("\n===== TICKET DATA ANALYSIS SYSTEM =====")
    print("1. Display all tickets")
    print("2. Add a new ticket")
    print("3. Search a ticket")
    print("4. Update a ticket")
    print("5. Delete a ticket")
    print("6. Summary report")
    print("7. NumPy demo")
    print("8. Pandas analysis")
    print("9. Data insights")
    print("10. Handle missing values")
    print("0. Exit")
    choice = input("Enter your choice: ").strip()
    if choice == "1":
        print_header("ALL TICKETS"); show_table(add_derived_columns(df))
    elif choice == "2":
        print_header("ADD NEW TICKET"); new_id = next_ticket_id(df)
        record = {"ticket_id": new_id, "customer_name": input("Customer name: "), "department": input("Department: "), "category": input("Category: "), "priority": input("Priority (Low/Medium/High/Urgent): "), "status": input("Status (Open/In Progress/Resolved/Closed): "), "channel": input("Channel (Email/Phone/Chat/Web): "), "city": input("City: "), "response_time": float(input("Response time in hours: ")), "resolution_time": float(input("Resolution time in hours: ")), "customer_rating": float(input("Customer rating (1-5): "))}
        df = add_record(df, record); print("Ticket added. Ticket ID:", new_id)
    elif choice == "3":
        print_header("SEARCH TICKET"); result = search_record(df, input("Enter ticket ID, customer name or category: ")); print(len(result), "record(s) found"); show_table(result)
    elif choice == "4":
        print_header("UPDATE TICKET"); ticket_id = int(input("Ticket ID to update: ")); field = input("Field (customer_name/department/category/priority/status/channel/city/response_time/resolution_time/customer_rating): ").strip(); value = input("New value: ")
        if field in ["response_time","resolution_time","customer_rating"]: value = float(value)
        df, done = update_record(df, ticket_id, {field:value}); print("Record updated." if done else "Ticket ID not found or field invalid.")
    elif choice == "5":
        print_header("DELETE TICKET"); df, done = delete_record(df, int(input("Ticket ID to delete: "))); print("Record deleted." if done else "Ticket ID not found.")
    elif choice == "6": summary_report(handle_missing_values(df))
    elif choice == "7": numpy_demo(handle_missing_values(df))
    elif choice == "8": pandas_demo(handle_missing_values(df))
    elif choice == "9": insights(handle_missing_values(df))
    elif choice == "10":
        df = handle_missing_values(df); save_data(df); print("Cleaned data saved to", FILE_PATH)
    elif choice == "0":
        print("Thank you. Goodbye!"); break
    else: print("Invalid choice, try again.")
    input("\nPress Enter to continue...")


Total records loaded: 60
First record: {'ticket_id': '2001', 'customer_name': 'Karan', 'department': 'General', 'category': 'Login Issue', 'priority': 'Medium', 'status': 'Open', 'channel': 'Web', 'city': 'Lucknow', 'response_time': '31', 'resolution_time': '84', 'customer_rating': '4'}
Cities (set): ['Delhi', 'Ghaziabad', 'Jaipur', 'Lucknow', 'Noida']
List methods demo: [125, 120, 115, 105, 101, 100]

===== TICKET DATA ANALYSIS SYSTEM =====
1. Display all tickets
2. Add a new ticket
3. Search a ticket
4. Update a ticket
5. Delete a ticket
6. Summary report
7. NumPy demo
8. Pandas analysis
9. Data insights
10. Handle missing values
0. Exit
Enter your choice: 1

                        ALL TICKETS                         
 ticket_id customer_name        department        category priority      status channel      city  response_time  resolution_time  customer_rating  priority_score response_category         sla_status
      2001         Karan           General     Login Issue   Medium  